# HealthBrowse Medical Model Lab — separate QLoRA experiment

This notebook develops a **separate adapter** for General and Clinicians. It does not change the live HealthBrowse model or call its API.

Start with **Runtime → Change runtime type → GPU**. You need access to the gated Meta Llama 3 repository and a Hugging Face read token saved in Colab Secrets under `HF_TOKEN` with notebook access enabled. Do not paste a token into code.

The default is a **one-step smoke test** using eight bundled, unreviewed exact-excerpt fixtures. Its weights are not a usable medical model. For a candidate run, first curate and review training and validation files using `healthbrowse_model_lab/DATASET.md`. No clinical validation or performance improvement is claimed by successful execution.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, datetime, uuid
REPO = Path('/content/healthbrowse-independent-model')
BRANCH = 'research/healthbrowse-medical-model'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/abby2395/medical_chatbot.git', str(REPO)], check=True)
if not (REPO / 'healthbrowse_model_lab/pyproject.toml').is_file():
    raise RuntimeError('This checkout does not contain the separate model lab. Use a fresh runtime and rerun this notebook.')
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
LAB = REPO / 'healthbrowse_model_lab'
os.chdir(LAB)
print('Separate source version:', SOURCE_COMMIT)


In [ ]:
# Keep the text-model stack independent of packages preinstalled in the Colab kernel.
import venv
ENV_DIR = Path('/content') / ('healthbrowse-train-' + sys.implementation.cache_tag)
venv.EnvBuilder(with_pip=False).create(ENV_DIR)
LAB_PYTHON = str(ENV_DIR / 'bin/python')
lab_env = os.environ.copy()
lab_env.pop('PYTHONPATH', None)
lab_env['PYTHONNOUSERSITE'] = '1'

def run_lab(*args):
    subprocess.run([LAB_PYTHON, *map(str, args)], cwd=LAB, env=lab_env, check=True)

# Target the venv directly; Colab's system Python may not ship ensurepip.
subprocess.run([sys.executable, '-m', 'pip', '--python', LAB_PYTHON,
                'install', '-e', '.[train,test]'], cwd=LAB, env=lab_env, check=True)
run_lab('-c', """import torch, transformers, accelerate, bitsandbytes
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from transformers.integrations import get_reporting_integration_callbacks
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
print('Trainer imports passed.', 'torch', torch.__version__, 'transformers', transformers.__version__)
""")


In [ ]:
from google.colab import userdata
run_lab('-m', 'hb_model_lab.cli', 'preflight')
run_lab('-c', "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime before training.'")
try:
    lab_env['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError("Enable notebook access to HF_TOKEN in Colab Secrets.") from exc


In [ ]:
run_lab('-m', 'unittest', 'discover', '-s', 'tests', '-v')


## Choose the run type

Leave `RUN_MODE = 'smoke'` for the first attempt; no uploads are needed. It checks that the real GPU can load the base model and save a new adapter.

Use `RUN_MODE = 'reviewed'` only after preparing the two JSONL files with valid rights and target-review records. The upload cell expects exactly `train.jsonl` and `validation.jsonl`. Their topic families and source pages must not overlap. The examples in this notebook are not a replacement for that curated dataset.


In [ ]:
RUN_MODE = 'smoke'  # 'smoke' or 'reviewed'
if RUN_MODE not in {'smoke', 'reviewed'}:
    raise ValueError('Choose smoke or reviewed')
from google.colab import drive, files
drive.mount('/content/drive')
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:6]
RUN_ROOT = Path('/content/drive/MyDrive/HealthBrowseModelLab') / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)
if RUN_MODE == 'smoke':
    TRAIN, VALIDATION = LAB / 'data/examples/train.jsonl', LAB / 'data/examples/validation.jsonl'
else:
    uploaded = files.upload()
    if set(uploaded) != {'train.jsonl', 'validation.jsonl'}:
        raise ValueError('Select exactly the reviewed train.jsonl and validation.jsonl files.')
    DATA = RUN_ROOT / 'data'
    DATA.mkdir()
    for filename, contents in uploaded.items():
        (DATA / filename).write_bytes(contents)
    del uploaded
    TRAIN, VALIDATION = DATA / 'train.jsonl', DATA / 'validation.jsonl'
run_lab('-m', 'hb_model_lab.cli', 'validate', '--train', TRAIN, '--validation', VALIDATION,
        *(['--examples'] if RUN_MODE == 'smoke' else []))
print('Run directory:', RUN_ROOT)


In [ ]:
run_lab('-m', 'hb_model_lab.cli', 'train', '--train', TRAIN, '--validation', VALIDATION,
        '--output', RUN_ROOT / 'model', *(['--smoke'] if RUN_MODE == 'smoke' else []))
MANIFEST = json.loads((RUN_ROOT / 'model/run-manifest.json').read_text())
print('Saved separately:', RUN_ROOT / 'model/adapter')


## After training

For the one-step smoke test, check the printed manifest for `"status": "trained"` and `"smoke_only": true`. For a reviewed candidate, freeze a held-out evaluation set and use the isolated environment for both runs:

```python
run_lab('-m', 'hb_model_lab.cli', 'evaluate', '--baseline', '--output', RUN_ROOT / 'eval-base')
run_lab('-m', 'hb_model_lab.cli', 'evaluate', '--adapter', RUN_ROOT / 'model/adapter',
        '--output', RUN_ROOT / 'eval-adapter')
run_lab('-m', 'hb_model_lab.cli', 'compare', '--baseline', RUN_ROOT / 'eval-base',
        '--adapter', RUN_ROOT / 'eval-adapter', '--output', RUN_ROOT / 'comparison.json')
```

The bundled benchmark runs 200 cases per evaluation. Inspect predictions and complete independent human review before drawing conclusions about answer quality.
